# Folds da validação cruzada

Gera uma única vez as 10 partições estratificadas usadas por todos os algoritmos e configurações, nas duas bases (com e sem `inativo`), e salva em `data/processed/folds.csv` (coluna `fold`: número do fold de teste de cada cliente, na ordem das linhas dos CSVs pré-processados).

O `StratifiedKFold` usa só o `y` para estratificar (o `X` entra apenas pelo número de linhas). Como as duas bases têm as mesmas 2850 linhas, na mesma ordem e com o mesmo `churn`, os folds valem para as duas.

In [1]:
import numpy as np
import pandas as pd

from comum import (FOLDS_CSV, N_FOLDS, SEMENTE, carregar_base, carregar_folds,
                   conferir_folds, gerar_folds)

print(f'SEMENTE = {SEMENTE} | N_FOLDS = {N_FOLDS}')

SEMENTE = 42 | N_FOLDS = 10


## Bases e classe

As duas bases precisam ter o mesmo `y`, linha a linha.

In [2]:
X_com, y_com = carregar_base(com_status=True)
X_sem, y_sem = carregar_base(com_status=False)

assert np.array_equal(y_com, y_sem)
assert X_com.drop(columns='inativo').equals(X_sem)
y = y_com
print(X_com.shape, X_sem.shape, f'churn: {y.sum()} ({y.mean():.2%})')

(2850, 16) (2850, 15) churn: 446 (15.65%)


## Geração e gravação

In [3]:
fold = gerar_folds(y)
conferir_folds(fold, y)

pd.DataFrame({'fold': fold}).to_csv(FOLDS_CSV, index=False)
print(f'salvo em {FOLDS_CSV.relative_to(FOLDS_CSV.parents[2])}')

salvo em data\processed\folds.csv


## Composição de cada fold de teste

Cada fold de teste tem ~285 clientes e 44 ou 45 churns (446 / 10 = 44,6), mantendo a proporção da base.

In [4]:
composicao = (pd.DataFrame({'fold': fold, 'churn': y})
              .groupby('fold')['churn']
              .agg(clientes='size', churns='sum', pct_churn='mean'))
composicao['pct_churn'] = (composicao['pct_churn'] * 100).round(2)
composicao

,clientes,churns,pct_churn
fold,,,
0,285,44,15.44
1,285,44,15.44
2,285,44,15.44
3,285,44,15.44
4,285,45,15.79
5,285,45,15.79
6,285,45,15.79
7,285,45,15.79
8,285,45,15.79


## Releitura

Os notebooks de modelagem usam `carregar_folds(y)`, que relê o arquivo e repete as conferências (tamanho, folds 0 a 9, estratificação e igualdade com os folds gerados pela `SEMENTE`).

In [5]:
folds = carregar_folds(y)

assert len(folds) == N_FOLDS
for treino, teste in folds:
    assert len(np.intersect1d(treino, teste)) == 0
    assert len(treino) + len(teste) == len(y)
assert np.array_equal(np.sort(np.concatenate([t for _, t in folds])), np.arange(len(y)))
print('folds relidos e conferidos:', [len(t) for _, t in folds])

folds relidos e conferidos: [285, 285, 285, 285, 285, 285, 285, 285, 285, 285]
